In [1]:
import pandas as pd

In [2]:
DATA_PATH = "../data/Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.shape

(7043, 21)

### Dimensiones del dataset

El dataset contiene 7.043 observaciones y 21 columnas.  
Una de ellas corresponde al identificador `customerID` y otra a la variable objetivo `Churn`.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [5]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [6]:
(df["TotalCharges"].astype(str).str.strip() == "").sum()

np.int64(11)

In [7]:
df[df["TotalCharges"].astype(str).str.strip() == ""][
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["customerID"].duplicated().sum()

np.int64(0)

In [10]:
df["customerID"].nunique()

7043

### Identificador de cliente

`customerID` presenta un valor único por observación y no contiene información predictiva generalizable.  
Por este motivo será excluido de las variables utilizadas para entrenar el modelo.

In [11]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [12]:
df["Churn"].value_counts(normalize=True).mul(100).round(2)

Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64

### Variable objetivo

La variable `Churn` presenta un desbalance moderado:

- No: 5.174 clientes (73,46%)
- Yes: 1.869 clientes (26,54%)

Debido a este desbalance, la evaluación del modelo no se basará únicamente en accuracy. Se incorporarán métricas que permitan evaluar adecuadamente la capacidad de distinguir la clase positiva.

In [13]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


customerID
customerID
7590-VHVEG    1
3791-LGQCY    1
6008-NAIXK    1
5956-YHHRX    1
5365-LLFYV    1
             ..
9796-MVYXX    1
2637-FKFSY    1
1552-AAGRX    1
4304-TSPVK    1
3186-AJIEK    1
Name: count, Length: 7043, dtype: int64

gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
OnlineBackup
No                     3088
Yes                    2429


In [14]:
df["SeniorCitizen"].value_counts()

SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64

In [15]:
df[["tenure", "MonthlyCharges"]].describe()

,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


In [16]:
total_charges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")

total_charges_numeric.describe()

count    7032.000000
mean     2283.300441
std      2266.771362
min        18.800000
25%       401.450000
50%      1397.475000
75%      3794.737500
max      8684.800000
Name: TotalCharges, dtype: float64

In [18]:
total_charges_numeric.isna().sum()

np.int64(11)

### Revisión de TotalCharges

`TotalCharges` aparece originalmente como una variable de texto debido a la presencia de 11 registros vacíos.

Estos 11 casos corresponden a clientes con `tenure = 0`, por lo que no se eliminarán del dataset. Durante la preparación de los datos, los espacios vacíos de `TotalCharges` serán interpretados como valores faltantes y tratados de forma reproducible.

In [19]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

print("Variables numéricas:", len(numeric_features))
print("Variables categóricas:", len(categorical_features))
print("Total predictores:", len(numeric_features) + len(categorical_features))

Variables numéricas: 3
Variables categóricas: 16
Total predictores: 19


In [20]:
pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
).round(3)

Churn,No,Yes
Contract,,
Month-to-month,0.573,0.427
One year,0.887,0.113
Two year,0.972,0.028


In [21]:
pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
).round(3)

Churn,No,Yes
InternetService,,
DSL,0.810,0.190
Fiber optic,0.581,0.419
No,0.926,0.074


In [22]:
pd.crosstab(
    df["PaymentMethod"],
    df["Churn"],
    normalize="index"
).round(3)

Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),0.833,0.167
Credit card (automatic),0.848,0.152
Electronic check,0.547,0.453
Mailed check,0.809,0.191


### Algunas relaciones con Churn

Se observan diferencias importantes en la proporción de abandono según algunas variables categóricas.

Los clientes con contrato mensual presentan una mayor proporción de churn que aquellos con contratos de uno o dos años. También se observa una mayor proporción de abandono entre clientes con servicio de fibra óptica y entre quienes utilizan cheque electrónico como medio de pago.

Estas diferencias son descriptivas y no implican causalidad, pero muestran que estas variables pueden aportar información útil al modelo.

In [23]:
df_numeric = df.copy()

df_numeric["TotalCharges"] = pd.to_numeric(
    df_numeric["TotalCharges"],
    errors="coerce"
)

df_numeric.groupby("Churn")[
    ["tenure", "MonthlyCharges", "TotalCharges"]
].mean().round(2)

,tenure,MonthlyCharges,TotalCharges
Churn,,,
No,37.57,61.27,2555.34
Yes,17.98,74.44,1531.80


### Variables numéricas y Churn

Los clientes que presentan churn tienen una antigüedad promedio menor que quienes permanecen en el servicio (17,98 frente a 37,57 meses).

También presentan un cargo mensual promedio más alto. En cambio, su cargo total acumulado es menor, lo que es consistente con una menor permanencia como clientes.

Estas relaciones son descriptivas y serán consideradas durante el modelamiento.

## Decisiones para el modelamiento

A partir de la exploración realizada se definen las siguientes decisiones:

- `Churn` será la variable objetivo del modelo de clasificación binaria.
- `customerID` será excluida, ya que corresponde únicamente a un identificador de cliente.
- Se utilizarán 19 variables predictoras.
- Los espacios vacíos de `TotalCharges` serán interpretados como valores faltantes al cargar los datos y tratados dentro del pipeline.
- Los 11 valores vacíos de `TotalCharges` corresponden a clientes con `tenure = 0` y serán tratados durante el preprocesamiento.
- `SeniorCitizen` será tratada como variable categórica, aunque originalmente esté representada mediante 0 y 1.
- Las variables categóricas serán codificadas mediante `OneHotEncoder`.
- Las variables numéricas serán tratadas dentro del pipeline de preprocesamiento.
- El preprocesamiento y el modelo quedarán integrados en un único `Pipeline` de scikit-learn para evitar diferencias entre entrenamiento e inferencia.
- Debido al desbalance de `Churn`, la evaluación no se basará únicamente en accuracy.